[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theochem/grid/blob/master/examples/Local_Grid.ipynb)

# Local Grids

This example on Formaldehyde shows how to use the local grid functionality to integrate a property over a small region of a large molecule.

A local grid contains only the grid points within a sphere around a chosen center. The points are found efficiently with a KDTree (`scipy.spatial.cKDTree`), which is built the first time `get_localgrid` is called and reused afterwards.

The format checkpoint file wavefunction is included in "examples/ch2o_q+0.fchk" and is read using the [IOData](https://github.com/theochem/iodata) package.

In [6]:
# # Install packages in Google Colab
# ! pip install qc-iodata
# ! pip install qc-grid
# ! pip install qc-gbasis


# # Download the example files
# from urllib.request import urlretrieve
# urlretrieve("https://raw.githubusercontent.com/theochem/grid/refs/heads/master/examples/ch2o_q%2B0.fchk", "ch2o_q+0.fchk")

First, build a molecular grid for Formaldehyde and evaluate the electron density on it, following the same construction as in the Molecular Grid example. A radial grid (Gauss-Legendre points mapped with a Becke transformation) is combined with atomic grids and Becke weights. The density is computed with the [gbasis](https://github.com/theochem/gbasis) package. Integrating it over the full grid should give 16, the number of electrons in Formaldehyde.

In [7]:
import numpy as np
from iodata import load_one
from grid.onedgrid import GaussLegendre
from grid.rtransform import BeckeRTransform
from grid.becke import BeckeWeights
from grid.molgrid import MolGrid
from gbasis.wrappers import from_iodata
from gbasis.evals.density import evaluate_density

mol = load_one("ch2o_q+0.fchk")

oned_grid = GaussLegendre(npoints=50)
radial_grid = BeckeRTransform(0.0, R=1.5).transform_1d_grid(oned_grid)

mol_grid = MolGrid.from_preset(
    atnums=mol.atnums,
    atcoords=mol.atcoords,
    rgrid=radial_grid,
    preset="fine",
    aim_weights=BeckeWeights(),
    store=True,
)

basis = from_iodata(mol)
rdm = mol.one_rdms["scf"]
dens = evaluate_density(rdm, basis, mol_grid.points)

print(f"Number of grid points: {len(mol_grid.points)}")
print(f"Integral of the density: {mol_grid.integrate(dens): .6f}")
print(f"Atomic coordinates:\n{mol.atcoords}")

Number of grid points: 16448
Integral of the density:  15.999988
Atomic coordinates:
[[ 2.27823914e+00  4.13899085e-07  3.12033662e-07]
 [ 1.01154892e-02  1.09802629e-07 -6.99333116e-07]
 [-1.09577141e+00  1.77311416e+00  1.42544321e-07]
 [-1.09577166e+00 -1.77311468e+00  2.44755133e-07]]


### Creating a local grid

The method `get_localgrid(center, radius)` returns a `LocalGrid` containing only the grid points within `radius` (in bohr) of `center`. Here the center is the position of atom 0, which is the Oxygen atom (atomic number 8).

The `indices` attribute of the local grid gives the positions of its points in the full molecular grid. This is how to get the density on the local grid, `dens[local.indices]`, and `local.integrate` then integrates it over this small region only. The result is the number of electrons within the sphere around the Oxygen atom.

In [8]:
print(f"Atomic numbers: {mol.atnums}")

center = mol.atcoords[0]   # coordinates of atom 0
radius = 2.0               # in bohr (atomic units)

local = mol_grid.get_localgrid(center, radius)

print(f"Points in the full grid : {len(mol_grid.points)}")
print(f"Points in the local grid: {len(local.points)}")
print(f"First few indices       : {local.indices[:5]}")

# Integrate the density over only the local region
local_dens = dens[local.indices]
print(f"Electrons within {radius} bohr of atom 0: {local.integrate(local_dens): .6f}")

Atomic numbers: [8 6 1 1]
Points in the full grid : 16448
Points in the local grid: 3375
First few indices       : [15190  6984  2514  2517  2515]
Electrons within 2.0 bohr of atom 0:  8.561305


### Effect of the radius

The size of the local grid is controlled by the radius. A small radius includes few grid points, so the integration is cheap but only captures the density near the center. As the radius grows, more points are included and the integrated density approaches the total number of electrons of Formaldehyde (16).

With `radius=np.inf`, the local grid contains every point of the molecular grid, so its integral matches the integral over the full grid computed above.

In [9]:
center = mol.atcoords[0]  # Oxygen atom

for radius in [1.0, 2.0, 3.0, 5.0, 8.0, np.inf]:
    local = mol_grid.get_localgrid(center, radius)
    electrons = local.integrate(dens[local.indices])
    print(f"radius = {radius:>4}  |  points = {len(local.points):>6}  |  electrons = {electrons: .6f}")

radius =  1.0  |  points =    934  |  electrons =  4.949394
radius =  2.0  |  points =   3375  |  electrons =  8.561305
radius =  3.0  |  points =   6234  |  electrons =  12.771719
radius =  5.0  |  points =  11638  |  electrons =  15.772143
radius =  8.0  |  points =  14331  |  electrons =  15.998261
radius =  inf  |  points =  16448  |  electrons =  15.999988
